In [5]:
from pathlib import Path
import json
import duckdb
import pyarrow.parquet as pq

# 다운로드한 파일들이 들어 있는 폴더로 변경
OUT = (Path.cwd().parent / "parquets" / "HI-Large").resolve()

print("현재 작업 위치:", Path.cwd())
print("검사할 경로:", OUT.resolve())
print("폴더 존재:", OUT.is_dir())

# 기존 HI-Large 전처리 결과 기준
EXPECTED_ROWS = {
    "transactions.parquet": 179_702_079,
    "duplicates.parquet": 150,
    "kyc.parquet": 2_116_168,
    "patterns.parquet": 137_936,
}

REQUIRED_COLUMNS = {
    "transactions.parquet": {
        "edge_id", "ts", "src", "dst", "paid", "received",
        "payment_currency", "receiving_currency", "payment_format",
        "duplicate_group_id", "is_laundering",
        "from_bank", "from_account", "to_bank", "to_account", "day",
    },
    "duplicates.parquet": {"edge_id", "duplicate_group_id", "group_size"},
    "kyc.parquet": {"node_id", "bank_id", "account", "entity_id"},
    "patterns.parquet": {
        "attempt_id", "pattern_type", "pattern_desc",
        "edge_id", "ts", "attempt_size", "unmatched",
    },
}

issues = []
readable = set()
actual_rows = {}

def verify(label, actual, expected):
    ok = actual == expected
    print(
        f"[{'OK' if ok else 'FAIL'}] {label}: "
        f"{actual} / 기대 {expected}",
        flush=True,
    )
    if not ok:
        issues.append(label)

def q(path):
    return "'" + str(path).replace("'", "''") + "'"

# 1. 모든 컬럼을 배치 단위로 실제 읽기
# 한 번에 전체 데이터를 pandas로 변환하지 않음
for name, expected in EXPECTED_ROWS.items():
    path = OUT / name
    print(f"\n검사 시작: {name}", flush=True)

    if not path.is_file():
        print("[FAIL] 파일 없음", flush=True)
        issues.append(f"{name}: 파일 없음")
        continue

    try:
        with pq.ParquetFile(path) as pf:
            missing = REQUIRED_COLUMNS[name] - set(pf.schema_arrow.names)
            if missing:
                raise ValueError(f"필수 컬럼 누락: {sorted(missing)}")

            rows = 0
            for batch in pf.iter_batches(
                batch_size=65_536,
                use_threads=False,
            ):
                batch.validate(full=True)
                rows += batch.num_rows

            verify(f"{name} 메타데이터 행 수", rows, pf.metadata.num_rows)

        actual_rows[name] = rows
        readable.add(name)
        verify(f"{name} 전체 읽기 행 수", rows, expected)
        print(
            f"전체 컬럼 읽기 완료 / {path.stat().st_size / 1e9:.3f} GB",
            flush=True,
        )

    except Exception as e:
        print(f"[FAIL] {name}: {e}", flush=True)
        issues.append(f"{name}: {e}")

# 2. 주요 집계 검사
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute("SET threads=2")

actual_summary = {}

try:
    if "transactions.parquet" in readable:
        row = con.execute(f"""
            SELECT
                count(*),
                min(ts),
                max(ts),
                sum(is_laundering::BIGINT),
                count(*) FILTER (WHERE src = dst),
                count(*) FILTER (
                    WHERE edge_id IS NULL OR ts IS NULL
                       OR src IS NULL OR dst IS NULL
                       OR is_laundering IS NULL
                ),
                count(*) FILTER (
                    WHERE duplicate_group_id IS NOT NULL
                      AND edge_id <> duplicate_group_id
                )
            FROM read_parquet({q(OUT / "transactions.parquet")})
        """).fetchone()

        actual_summary.update(
            rows_kept=row[0],
            ts_min=str(row[1]),
            ts_max=str(row[2]),
            laundering_rows=row[3],
            self_transfers=row[4],
        )

        print("\n거래 내용 검사", flush=True)
        verify("시작 시각", str(row[1]), "2022-08-01 00:00:00")
        verify("마지막 시각", str(row[2]), "2023-01-12 16:49:00")
        verify("자금세탁 거래 수", row[3], 225_546)
        verify("셀프이체 수", row[4], 10_895_211)
        verify("핵심 컬럼 NULL 행 수", row[5], 0)
        verify("제거돼야 할 중복 거래 잔존", row[6], 0)

    if "duplicates.parquet" in readable:
        groups, invalid = con.execute(f"""
            SELECT
                count(DISTINCT duplicate_group_id),
                count(*) FILTER (
                    WHERE edge_id IS NULL
                       OR duplicate_group_id IS NULL
                       OR edge_id = duplicate_group_id
                       OR group_size IS NULL OR group_size <= 1
                )
            FROM read_parquet({q(OUT / "duplicates.parquet")})
        """).fetchone()
        actual_summary.update(
            rows_duplicate_removed=actual_rows["duplicates.parquet"],
            duplicate_groups=groups,
        )
        verify("중복 그룹 수", groups, 150)
        verify("중복 파일의 잘못된 행", invalid, 0)

    if "kyc.parquet" in readable:
        missing = con.execute(f"""
            SELECT count(*) FILTER (WHERE entity_id IS NULL)
            FROM read_parquet({q(OUT / "kyc.parquet")})
        """).fetchone()[0]
        actual_summary.update(
            nodes=actual_rows["kyc.parquet"],
            nodes_without_account_row=missing,
        )
        verify("계좌 정보 미매칭 수", missing, 0)

    if "patterns.parquet" in readable:
        attempts, unmatched, null_flags = con.execute(f"""
            SELECT
                count(DISTINCT attempt_id),
                count(*) FILTER (WHERE unmatched = true OR edge_id IS NULL),
                count(*) FILTER (WHERE unmatched IS NULL OR attempt_id IS NULL)
            FROM read_parquet({q(OUT / "patterns.parquet")})
        """).fetchone()
        actual_summary.update(
            pattern_rows=actual_rows["patterns.parquet"],
            pattern_attempts=attempts,
            pattern_rows_unmatched=unmatched,
        )
        verify("패턴 시도 수", attempts, 16_467)
        verify("거래 미매칭 패턴 행", unmatched, 0)
        verify("패턴 핵심 컬럼 NULL 행", null_flags, 0)

except Exception as e:
    print(f"[FAIL] 내용 검사 중 오류: {e}", flush=True)
    issues.append(f"내용 검사: {e}")
finally:
    con.close()

# 3. summary.json과 실제 파일 비교
print("\nsummary.json 검사", flush=True)
try:
    summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8-sig"))
    verify("데이터셋", summary.get("dataset"), "HI-Large")
    for key, actual in actual_summary.items():
        verify(f"summary.{key}", summary.get(key), actual)
except Exception as e:
    print(f"[FAIL] summary.json: {e}", flush=True)
    issues.append(f"summary.json: {e}")

print("\n" + "=" * 60)
if issues:
    print(f"확인 필요한 항목: {len(issues)}개")
    for issue in issues:
        print(" -", issue)
else:
    print("PASS: 전체 파일 읽기·행 수·주요 집계·summary 일치 검사 통과")

현재 작업 위치: /workspace/money_laundry_ai_/code
검사할 경로: /workspace/money_laundry_ai_/parquets/HI-Large
폴더 존재: True

검사 시작: transactions.parquet
[OK] transactions.parquet 메타데이터 행 수: 179702079 / 기대 179702079
[OK] transactions.parquet 전체 읽기 행 수: 179702079 / 기대 179702079
전체 컬럼 읽기 완료 / 4.799 GB

검사 시작: duplicates.parquet
[OK] duplicates.parquet 메타데이터 행 수: 150 / 기대 150
[OK] duplicates.parquet 전체 읽기 행 수: 150 / 기대 150
전체 컬럼 읽기 완료 / 0.000 GB

검사 시작: kyc.parquet
[OK] kyc.parquet 메타데이터 행 수: 2116168 / 기대 2116168
[OK] kyc.parquet 전체 읽기 행 수: 2116168 / 기대 2116168
전체 컬럼 읽기 완료 / 0.030 GB

검사 시작: patterns.parquet
[OK] patterns.parquet 메타데이터 행 수: 137936 / 기대 137936
[OK] patterns.parquet 전체 읽기 행 수: 137936 / 기대 137936
전체 컬럼 읽기 완료 / 0.001 GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


거래 내용 검사
[OK] 시작 시각: 2022-08-01 00:00:00 / 기대 2022-08-01 00:00:00
[OK] 마지막 시각: 2023-01-12 16:49:00 / 기대 2023-01-12 16:49:00
[OK] 자금세탁 거래 수: 225546 / 기대 225546
[OK] 셀프이체 수: 10895211 / 기대 10895211
[OK] 핵심 컬럼 NULL 행 수: 0 / 기대 0
[OK] 제거돼야 할 중복 거래 잔존: 0 / 기대 0
[OK] 중복 그룹 수: 150 / 기대 150
[OK] 중복 파일의 잘못된 행: 0 / 기대 0
[OK] 계좌 정보 미매칭 수: 0 / 기대 0
[OK] 패턴 시도 수: 16467 / 기대 16467
[OK] 거래 미매칭 패턴 행: 0 / 기대 0
[OK] 패턴 핵심 컬럼 NULL 행: 0 / 기대 0

summary.json 검사
[OK] 데이터셋: HI-Large / 기대 HI-Large
[OK] summary.rows_kept: 179702079 / 기대 179702079
[OK] summary.ts_min: 2022-08-01 00:00:00 / 기대 2022-08-01 00:00:00
[OK] summary.ts_max: 2023-01-12 16:49:00 / 기대 2023-01-12 16:49:00
[OK] summary.laundering_rows: 225546 / 기대 225546
[OK] summary.self_transfers: 10895211 / 기대 10895211
[OK] summary.rows_duplicate_removed: 150 / 기대 150
[OK] summary.duplicate_groups: 150 / 기대 150
[OK] summary.nodes: 2116168 / 기대 2116168
[OK] summary.nodes_without_account_row: 0 / 기대 0
[OK] summary.pattern_rows: 137936 / 기대 137936
[OK] summ